In [8]:
# Import packages and initialize Earth Engine

import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns

ee.Authenticate()
ee.Initialize(project='ee-ivanburgov666')

### Parameters

In [9]:
EXPORT_FOLDER = 'GEMLST_MODIS'
TILE_SCALE = 8
SCALE_M = 1000


### Initialize variables: Mask, extraction points and time frame 

In [10]:
date_start = '2002-07-04'
date_end = '2025-12-31'

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

poi = ee.FeatureCollection("projects/ee-ivanburgov666/assets/randomGR5km_masked_260508")


In [11]:
# Mask data based on quality flags analogously to extraction script from GEM stations. 
def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskQualityDaytime(image):
    qa = image.select('QC_Day')
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)
    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)
    return image.updateMask(mask)

def maskQualityNighttime(image):
    qa = image.select('QC_Night')
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)
    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)
    return image.updateMask(mask)

def maskViirs(image):
    '''Function to filter VIIRS LST data based on quality flag.'''
    qa = image.select('QC')
    bits01Mask = bitwiseExtract(qa, 0, 1).eq(0); 
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    bit1213Mask = bitwiseExtract(qa, 12, 13).gte(2)
    bit1415Mask = bitwiseExtract(qa, 14, 15).gte(2)
    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit1213Mask).And(bit1415Mask)
    return image.updateMask(mask)


In [12]:
# Load MODIS Terra and Aqua data, apply quality control and conversion functions
def lst_mod_day(image):
    'Terra Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
    qa_day = image.select('QC_Day').rename('MOD_QA_Day')
    overfly = image.select('Day_view_time').multiply(0.1).rename('time')
    # Keep only the converted value, QC and overfly-time bands.
    return image.addBands(lst_day).addBands(qa_day).addBands(overfly).select(['MOD_LST_Day', 'MOD_QA_Day', 'time'])

def lst_mod_night(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
    qa_night = image.select('QC_Night').rename('MOD_QA_Night')
    overfly = image.select('Night_view_time').multiply(0.1).rename('time')
    # Keep only the converted value, QC and overfly-time bands.
    return image.addBands(lst_night).addBands(qa_night).addBands(overfly).select(['MOD_LST_Night', 'MOD_QA_Night', 'time'])

def lst_myd_day(image):
    'Aqua Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
    qa_day = image.select('QC_Day').rename('MYD_QA_Day')
    overfly = image.select('Day_view_time').multiply(0.1).rename('time')
    # Keep only the converted value, QC and overfly-time bands.
    return image.addBands(lst_day).addBands(qa_day).addBands(overfly).select(['MYD_LST_Day', 'MYD_QA_Day', 'time'])

def lst_myd_night(image):
    'Aqua Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
    qa_night = image.select('QC_Night').rename('MYD_QA_Night')
    overfly = image.select('Night_view_time').multiply(0.1).rename('time')
    # Keep only the converted value, QC and overfly-time bands.
    return image.addBands(lst_night).addBands(qa_night).addBands(overfly).select(['MYD_LST_Night', 'MYD_QA_Night', 'time'])

MOD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Day_1km', 'QC_Day', 'Day_view_time'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_mod_day)
)

MOD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night', 'Night_view_time'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_mod_night)
)

MYD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Day_1km', 'QC_Day', 'Day_view_time'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_myd_day)
)

MYD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night', 'Night_view_time'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_myd_night)
)

imgTerraD = MOD11A1Daytime               
imgTerraN = MOD11A1Nighttime
imgAquaD = MYD11A1Daytime
imgAquaN = MYD11A1Nighttime

# Print the number of images in imgTerraD
# print('Number of images in imgTerraD:', imgTerraD.size().getInfo())

In [13]:
def add_poi_id(feature):
    poi_id = feature.get('object_id')
    cls = feature.get('class')
    return ee.Feature(feature).set({'object_id': poi_id, 'class': cls})

poi_with_id = poi.map(add_poi_id)

def extract_collection_at_poi(collection):
    def sample_image(img):
        date = ee.String(img.date().format('YYYY-MM-dd'))
        sampled = img.sampleRegions(
            collection=poi_with_id,
            properties=['object_id', 'class'],
            scale=SCALE_M,
            tileScale=TILE_SCALE,
            geometries=False,
        )
        return sampled.map(
            lambda f: ee.Feature(f).set(
                {
                    'object_id': f.get('object_id'),
                    'class': f.get('class'),
                    'date': date,
                    'sample_key': ee.String(f.get('object_id')).cat('_').cat(ee.String(f.get('class'))).cat('_').cat(date),
                }
            )
        )

    return ee.FeatureCollection(collection.map(sample_image).flatten())


poiLST_terraD = extract_collection_at_poi(imgTerraD)
poiLST_terraN = extract_collection_at_poi(imgTerraN)
poiLST_aquaD = extract_collection_at_poi(imgAquaD)
poiLST_aquaN = extract_collection_at_poi(imgAquaN)

# # Create a dataframe to preview results
# df_terra = geemap.ee_to_df(poiLST_terra).head()

# # df_terra = geemap.ee_to_df(poiLST_terra)
# metadata_columns = ['object_id', 'class', 'date', 'sample_key']
# ordered_columns = metadata_columns + [column for column in df_terra.columns if column not in metadata_columns]
# df_terra = df_terra[ordered_columns]
# print(df_terra.head())

sensors = {
    'TerraD': poiLST_terraD,
    'TerraN': poiLST_terraN,
   'AquaD': poiLST_aquaD,
   'AquaN': poiLST_aquaN
}


### Export

In [14]:
def export(sensor_name, sensor_value):
    description = f'GEMLST_{sensor_name}_LST_POI_reduced'

    selector_map = {
        'TerraD': ['object_id', 'class', 'date', 'MOD_LST_Day', 'MOD_QA_Day', 'time'],
        'TerraN': ['object_id', 'class', 'date', 'MOD_LST_Night', 'MOD_QA_Night', 'time'],
        'AquaD': ['object_id', 'class', 'date', 'MYD_LST_Day', 'MYD_QA_Day', 'time'],
        'AquaN': ['object_id', 'class', 'date', 'MYD_LST_Night', 'MYD_QA_Night', 'time'],
    }
    selectors = selector_map[sensor_name]

    task = ee.batch.Export.table.toDrive(
        collection=sensor_value,
        description=description,
        fileFormat='CSV',
        # projection='EPSG:3413', # kept native MODIS projection to accellerate sampling
        folder=EXPORT_FOLDER,
        selectors=selectors,
    )
    task.start()
    print(f'Started export task: {description}')


for sensor_name, sensor_values in sensors.items():
    export(sensor_name, sensor_values)
print('All export tasks started.')


Started export task: GEMLST_TerraD_LST_POI_reduced
Started export task: GEMLST_TerraN_LST_POI_reduced
Started export task: GEMLST_AquaD_LST_POI_reduced
Started export task: GEMLST_AquaN_LST_POI_reduced
All export tasks started.
